# AVO and Gassmann: testing a DHI

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosesfalowo/ogbonlab-notebooks/blob/main/notebooks/ssm_avo_gassmann_dhi.ipynb)


Generated by [OgbonLab](https://ogbonlab.com), Synthetic Seismic Modeling course.

**Assumptions**: Course defaults. Edit the parameters cell and run again; every value is yours to change.

**Workflow**:
1. 6.3 Zoeppritz and Shuey
2. 6.5 Synthetic AVO Gathers
3. 6.6 Acoustic or Elastic?

In [ ]:
# Ask Ogbon starter: Test a direct hydrocarbon indicator
# Generated by OgbonLab (ogbonlab.com), Synthetic Seismic Modeling course.
"""AVO of a direct hydrocarbon indicator: the top of a sand under shale, brine against hydrocarbon.

Follows Sections 6.3 (Zoeppritz and Shuey), 6.5 (Synthetic AVO Gathers) and 6.6
(Acoustic or Elastic?); the exact coefficient is Aki and Richards (1980).
Units: velocity m/s, density g/cc, angle degrees; coefficients have none.
Assumptions and limits:
- One plane, welded interface between isotropic elastic half-spaces: plane
  waves, no wavelet, no tuning, no overburden loss (Section 6.5 adds those).
- The intercept and gradient are a least-squares fit of Shuey's two-term form
  over 0 to MAX_ANGLE_DEG; past about 30 degrees the gradient absorbs the third term.
- The acoustic line keeps the density and drops the shear waves (Section 6.6).
- The field: Gassmann swaps the fill of the brine sand; the trap of the field earth
  holds that sand above a flat contact and brine below. Its near and far stacks use
  the linear form of Aki and Richards, sample by sample; the curves are exact.

It prints the numbers Figure 12.5 computes for the same answers as name = value
lines; then it builds the field earth every starter shares, prints what it measures
there as [field] name = value lines, and draws one figure. Python 3.9+, numpy and
matplotlib; the earth's random numbers are seeded, so every run prints the same.
Edit the parameters and run again.
"""
import numpy as np
import matplotlib.pyplot as plt

# --- parameters from your interview ---
BASELINE = 'approx'   # your brine baseline: 'logs', 'approx' or 'no'; with 'logs', put your sand in ROCKS
SANDS = ['gas sand']  # the sands to test against the brine sand
MAX_ANGLE_DEG = 35    # largest usable incidence angle (degrees)

In [ ]:
# --- the book's constants (not interview answers) ---
# the book's standard rocks (SSM.STANDARD, Section 3.3): P velocity (m/s), S velocity (m/s), density (g/cc)
ROCKS = {
    'shale': (2800, 1241.3793103448277, 2.2521031547808517),
    'brine sand': (2889.973997793982, 1504.4719169684909, 2.2545487499999997),
    'oil sand': (2543.1993596875604, 1529.5017037311773, 2.1813626422963988),
    'gas sand': (2341.92404704112, 1582.4208998550723, 2.0379042775100316)
}
# Gassmann on the book's soft sand (Section 9.3): porosity, quartz (GPa, g/cc), and the pore fills at
# the book's anchor, 50 C and 25 MPa (Batzle and Wang 1992): bulk modulus (GPa), density (g/cc)
PHI = 0.25
K_MIN, RHO_MIN = 36.6, 2.65
FILLS = {'brine sand': (3.0243945863581994, 1.068195), 'oil sand': (1.0684952731950532, 0.7754505691855965),
         'gas sand': (0.06061222613279554, 0.20161711004012717)}
NEAR_DEG, FAR_DEG = 8.0, 30.0   # the field's angle stacks (degrees)
F_HZ, DT_MS, T_MAX_MS, SNR = 30.0, 2.0, 1000.0, 8.0   # their Ricker peak (Hz), sample and length (ms), signal-to-noise

def report(name, value, fmt='%.6f'):
    """Print one result as name = value, the line Figure 12.5 checks against the book's kernel."""
    print(('%s = ' + fmt) % (name, value))

def style(ax, title, xlabel, ylabel, grid=True):
    """The starters' one look: a title, both axes labelled with their units, a light grid."""
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if grid:
        ax.grid(True, alpha=0.3)

def zoeppritz_pp(top, bot, theta_deg):
    """Exact P-P coefficient of a welded interface, Aki and Richards (1980), 5.39 and 5.40 (Section 6.3)."""
    # top and bot are (vp, vs, density) in any one set of units; below the critical angle the result is real
    a1, b1, r1 = top
    a2, b2, r2 = bot
    i1 = np.radians(theta_deg)
    p = np.sin(i1) / a1                               # ray parameter
    p2 = p * p
    def ccos(s):
        q = 1 - s * s
        return complex(np.sqrt(q), 0.0) if q >= 0 else complex(0.0, np.sqrt(-q))
    a = r2 * (1 - 2 * b2 * b2 * p2) - r1 * (1 - 2 * b1 * b1 * p2)
    b = r2 * (1 - 2 * b2 * b2 * p2) + 2 * r1 * b1 * b1 * p2
    c = r1 * (1 - 2 * b1 * b1 * p2) + 2 * r2 * b2 * b2 * p2
    d = 2 * (r2 * b2 * b2 - r1 * b1 * b1)
    I1 = np.cos(i1) / a1
    J1 = ccos(p * b1) / b1
    I2 = ccos(p * a2) / a2
    J2 = ccos(p * b2) / b2
    E = b * I1 + c * I2
    F = b * J1 + c * J2
    G = a - d * I1 * J2
    H = a - d * I2 * J1
    D = E * F + G * H * p2
    return (((b * I1 - c * I2) * F - (a + d * I1 * J2) * H * p2) / D).real

def acoustic_pp(top, bot, theta_deg):
    """Coefficient of two fluids that keep their density and have no shear waves (Section 6.6)."""
    t = np.radians(theta_deg)
    c2 = np.sqrt(1 - (np.sin(t) * bot[0] / top[0]) ** 2 + 0j)
    z1, z2 = top[0] * top[2], bot[0] * bot[2]
    return ((z2 * np.cos(t) - z1 * c2) / (z2 * np.cos(t) + z1 * c2)).real

# --- the Ogbon field earth: one seeded 2D earth that every starter shares ---
# In geological order: thin sand and shale beds with a regional dip, folded into an
# asymmetric anticline and cut by normal faults, the largest a listric growth fault
# (its throw grows with depth and its hanging wall rolls over into it); uplift, and an
# angular unconformity that truncates the crest; the reservoir sand subcrops beneath
# it, wedging out against it, shaling out to the left, sealed by a shale cap and
# holding gas above a flat contact; then a draped overburden with a channel. Each
# cell's velocity follows the
# compaction trend at its deepest burial (the rocks under the unconformity were once
# buried deeper), sands soft when shallow and hard when deep; Gardner's density; the
# mudrock and sandstone lines for S. Units: m, m/s, g/cc; times in ms.
FAULTS = [(0.66, 1.0, 1.4, 380.0), (0.80, -1.0, 0.6, 470.0), (0.22, -1.0, 0.5, 560.0)]   # (x / length, side, size, tip m)
COMPACTION = (1900.0, 1.4)                             # P velocity = 1900 m/s + 1.4 per metre of burial, before lithology
K_RES = 9                                              # the reservoir: bed 9 of the deep package, 70 m thick
SEAL_M = 30.0                                          # the shale cap on the reservoir's top: 30 m, and never under two cells

def field_earth(nx=240, nz=320, dx=15.0, dz=5.0, seed=7, crest=0.45, fold=1.0, dip=1.0, faults=None, throw=1.0,
                unconformity=True, seal=None, upper=None, lower=None, column_m=130.0):
    """The field earth on nz by nx cells of dz by dx (m). seal, upper and lower are (vp, vs, density) of the shale on the
    unconformity and of the reservoir above and below its flat contact, or None for the compaction trend's own rocks."""
    rng = np.random.default_rng(seed)
    x, z = np.arange(nx) * dx, np.arange(nz) * dz
    xn, zz, sc = x / x[-1], z[:, None] + 0.0 * x, nz * dz / 1600.0     # sc: the geometry scales with the grid's depth
    hump = np.exp(-0.5 * ((xn - crest) / np.where(xn < crest, 0.13, 0.25)) ** 2)   # the anticline, steep to the left
    swell = 1.0 + 0.15 * np.sin(2 * np.pi * (0.8 * xn + 0.37 * np.arange(26)[:, None]))   # each bed thickens and thins
    thick = sc * (24.0 + 22.0 * rng.random(26))[:, None] * swell
    thick[K_RES] = sc * 70.0
    beds = sc * 470.0 + np.cumsum(thick, axis=0) - thick + sc * (170.0 * dip * xn - 300.0 * fold * hump)
    if faults is not None:
        fs = [(rng.uniform(0.12, 0.88), rng.choice([-1.0, 1.0]), rng.uniform(0.4, 1.4), rng.uniform(300.0, 600.0))
              for k in range(faults)]
    fault = np.zeros((nz, nx), bool)
    for x0, side, size, tip in (FAULTS if faults is None else fs):
        tip, a = tip * sc, 0.2 / (1200.0 * sc) ** 1.3       # the plane: x / length = x0 + side a (depth - tip) to the 1.3, listric
        plane = np.where(side * (xn - x0) > 0, tip + (np.abs(xn - x0) / a) ** (1 / 1.3), -np.inf)   # its depth at each x
        for k in range(26):
            g = throw * size * sc * 90.0 * np.clip((beds[k] - tip) / (700.0 * sc), 0.0, 1.6)      # growth: deeper beds slip more
            xk = x0 + side * a * np.maximum(beds[k] - tip, 0.0) ** 1.3                          # where the plane cuts the bed
            g = g * (1.0 + 0.6 * np.exp(-np.abs(xn - xk) / 0.04))                                # the rollover into the fault
            beds[k] = beds[k] + np.clip(plane - beds[k], 0.0, g)       # the hanging wall drops; in the heave the bed lies on the plane
        ix = np.rint((x0 + side * a * np.maximum(z - tip, 0.0) ** 1.3) * (nx - 1)).astype(int)
        on = (z > tip) & (ix >= 0) & (ix < nx)
        fault[np.nonzero(on)[0], ix[on]] = True
    beds = np.maximum.accumulate(beds, axis=0)
    if unconformity:                                   # cut just under the reservoir's crest, so it subcrops on both flanks
        unc = beds[K_RES + 1, np.argmax(hump)] + sc * (15.0 + 160.0 * dip * (xn - crest)
                                                       + 10.0 * np.sin(2 * np.pi * (1.3 * xn + 0.2)))
    else:
        unc = beds[0] - sc * 30.0                      # no erosion: the overburden lies on the beds
    fault &= zz > unc                                  # the faults are older than the unconformity
    fr = np.cumsum(0.6 + rng.random(13))
    fr = (fr - fr[0]) / (fr[-1] - fr[0])
    over = [sc * 40.0 + (unc.min() - sc * 90.0) * f + 0.85 * f * f * (unc - unc.min()) - sc * 25.0 * f * hump for f in fr]
    tops = np.maximum.accumulate(np.vstack(over + list(np.maximum(beds, unc))), axis=0)
    lay = (zz[:, None, :] >= tops[None, :, :]).sum(axis=1)                    # layer 13 is the seal; bed k is layer 14 + k
    sand = np.where(lay < 13, rng.random(41)[lay] < 0.45, (lay - 14) % 2 == 1) & (lay > 0) & (lay != 13)
    chan = (zz >= over[5]) & (zz < over[5] + sc * 45.0 * np.exp(-0.5 * ((xn - 0.7) / 0.04) ** 2))
    res = lay == 14 + K_RES
    top = np.where(res, zz, np.inf).min(axis=0)
    cap = (zz < top) & (zz >= top - max(SEAL_M * sc, 2.0 * dz))   # the seal: shale on the sand's top, under a bed or the unconformity
    sand = np.where(cap, False, sand)
    burial = zz + np.where(lay >= 14, sc * 150.0 * hump, 0.0)                # uplift: the deep rocks were buried deeper
    vary = 1.0 + 0.035 * np.clip(rng.standard_normal(41), -1.2, 1.2)        # bed to bed variety
    vp = (COMPACTION[0] + COMPACTION[1] * burial) * vary[lay]
    vp = np.where(sand | chan, vp * (0.86 + 0.1 * np.minimum(burial / 1000.0, 1.8)), vp)
    vp = np.where(chan, vp * 0.92, vp)                 # the channel cuts a horizon and is filled with a soft sand
    sand |= chan
    vs = np.where(sand, 0.804 * vp - 856.0, (vp - 1360.0) / 1.16)            # the sandstone and mudrock lines (Castagna)
    den = 1.741 * (vp / 1000.0) ** 0.25                                      # Gardner
    ntg = np.clip((xn - 0.12) / 0.12, 0.0, 1.0)                              # the reservoir shales out to the left
    contact = float(top.min()) + column_m * sc         # the flat contact, column_m under the shallowest sand
    for rock, on in ((seal, (lay == 13) | cap), (upper, res & (zz < contact)), (lower, res & (zz >= contact))):
        if rock is not None:
            w = 1.0 if rock is seal else ntg
            vp, vs, den = [np.where(on, w * rock[i] + (1 - w) * q, q) for i, q in enumerate((vp, vs, den))]
    return dict(x=x, z=z, vp=vp, vs=vs, den=den, lay=lay, sand=sand, res=res, fault=fault, unc=unc, top=top,
                contact=contact, ntg=ntg, dx=dx, dz=dz, chan=chan, cap=cap, gas=res & (zz < contact) & (ntg > 0.5))

def field_time(m, dt_ms, t_max_ms):
    """Each trace from depth to two-way time by vertical rays: every cell array of m on a dt_ms grid to t_max_ms."""
    t_top = 2000.0 * np.cumsum(m['dz'] / m['vp'], axis=0) - 2000.0 * m['dz'] / m['vp']
    t = np.arange(0.0, t_max_ms, dt_ms)
    pick = np.array([np.searchsorted(t_top[:, i], t, 'right') - 1 for i in range(t_top.shape[1])]).T
    cols = np.arange(t_top.shape[1])[None, :]
    out = dict((k, m[k][pick, cols]) for k in ('vp', 'vs', 'den', 'lay', 'sand', 'res', 'gas'))
    seen = np.vstack([0 * cols, np.cumsum(m['fault'], axis=0)])                 # fault cells above each row
    after = np.maximum(np.vstack([pick[1:], pick[-1:] + 1]), pick + 1)
    out['fault'] = seen[np.minimum(after, seen.shape[0] - 1), cols] > seen[pick, cols]   # a fault cell inside the sample
    out['t'], out['t_top'] = t, t_top
    return out

def reflectivity(ai):
    """Normal-incidence coefficient at the top of every sample, down axis 0 (Section 2.1)."""
    rc = np.zeros_like(ai)
    rc[1:] = (ai[1:] - ai[:-1]) / (ai[1:] + ai[:-1])
    return rc

def q_peak_hz(f0, t_s, q):
    """Peak (Hz) of a Ricker of peak f0 after t_s seconds through constant Q: the root of 2 f f / (f0 f0) + (pi t / Q) f = 2."""
    b = np.pi * t_s / q
    return (-b + np.sqrt(b * b + 16.0 / f0 ** 2)) * f0 ** 2 / 4.0

def field_image(rc, dt_ms, f0, q=80.0, snr=None, seed=11):
    """Every trace convolved with a zero-phase Ricker whose peak falls with time as constant q says, plus coloured noise."""
    nt, dt = rc.shape[0], dt_ms * 1e-3
    f = np.fft.rfftfreq(2 * nt, dt)[:, None]
    def band(fp):                                      # the Ricker's spectrum (Section 1.1), over dt for a sum of samples
        return 2.0 / np.sqrt(np.pi) * f ** 2 / fp ** 3 * np.exp(-(f / fp) ** 2) / dt
    def filt(a, fp):
        return np.fft.irfft(np.fft.rfft(a, 2 * nt, axis=0) * band(fp), 2 * nt, axis=0)[:nt]
    fp = q_peak_hz(f0, np.arange(nt) * dt, q)
    knots = np.linspace(fp.min(), fp.max(), 4 if fp.max() - fp.min() > 1e-9 else 1)
    step = max((fp.max() - fp.min()) / 3.0, 1e-9)
    out = sum(np.clip(1.0 - np.abs(fp - k) / step, 0.0, 1.0)[:, None] * filt(rc, k) for k in knots)
    if snr:
        noise = filt(np.random.default_rng(seed).standard_normal(rc.shape), fp.mean())
        noise = (noise + np.roll(noise, 1, 1) + np.roll(noise, -1, 1)) / 3.0          # coherent over a few traces
        out = out + noise * np.sqrt((out ** 2).mean() / (noise ** 2).mean()) / snr
    return out

def fieldnote(name, value, fmt='%.3f'):
    """Print a number of the field earth as [field] name = value: computed here, not by Figure 12.5."""
    print(('[field] %s = ' + fmt) % (name, value))

def show(ax, a, x_m, y, title, ylabel='two-way time (ms)', clip=None):
    """A section as an image, black for positive, clipped at clip (or at its 99th percentile): km across, time or depth down."""
    c = clip if clip else np.percentile(np.abs(a), 99.0) + 1e-12
    ax.imshow(a, cmap='gray_r', vmin=-c, vmax=c, aspect='auto', extent=[x_m[0] / 1e3, x_m[-1] / 1e3, y[-1], y[0]])
    style(ax, title, 'distance (km)', ylabel, grid=False)

def critical_angle_deg(top, bot):
    """The P wave's critical angle (degrees) at the interface, or 90 when the lower rock is not faster."""
    return float(np.degrees(np.arcsin(top[0] / bot[0]))) if bot[0] > top[0] else 90.0

def validate(max_angle_deg, sands, rocks):
    """Stop with a clear message on a parameter the model cannot use."""
    for name in ['brine sand'] + list(sands):
        if name not in rocks:
            raise ValueError('unknown sand %r: ROCKS has %s' % (name, ', '.join(rocks)))
        crit = critical_angle_deg(rocks['shale'], rocks[name])
        if not 0 < max_angle_deg < crit:
            raise ValueError('MAX_ANGLE_DEG must lie between 0 and %.1f, the critical angle under the %s' % (crit, name))

def avo_curve(top, bot, angles_deg):
    """The exact coefficient at each angle."""
    return np.array([zoeppritz_pp(top, bot, th) for th in angles_deg])

def intercept_gradient(r, angles_deg):
    """Least-squares A and B of R = A + B sin(theta) squared (Shuey's two-term form, Section 6.3)."""
    s2 = np.sin(np.radians(angles_deg)) ** 2
    B, A = np.polyfit(s2, r, 1)
    return A, B

def report_sand(key, r, A, B):
    """The four numbers of one curve: normal incidence, the largest angle, the intercept, the gradient."""
    report(key + '_r0', r[0])
    report(key + '_r_max', r[-1])
    report(key + '_intercept', A)
    report(key + '_gradient', B)

def gassmann(rock, fill_from, fill_to):
    """Gassmann (1951): (vp, vs, density) of the sand with its pore fill swapped; frame and shear modulus kept (Section 9.3)."""
    vp, vs, den = rock[0] / 1e3, rock[1] / 1e3, rock[2]              # km/s, so the moduli are in GPa
    g = den * vs * vs
    k = den * vp * vp - 4.0 * g / 3.0
    def part(k_fill):
        return k_fill / (PHI * (K_MIN - k_fill))
    q = k / (K_MIN - k) - part(fill_from[0]) + part(fill_to[0])
    k2, den2 = K_MIN * q / (1 + q), den + PHI * (fill_to[1] - fill_from[1])
    return 1e3 * np.sqrt((k2 + 4.0 * g / 3.0) / den2), 1e3 * np.sqrt(g / den2), den2

def aki_richards(t, theta_deg):
    """The linear P-P coefficient at the top of every sample at one angle (Aki and Richards 1980, Section 6.3)."""
    th = np.radians(theta_deg)
    a, b, d = [(t[k][1:] + t[k][:-1]) / 2 for k in ('vp', 'vs', 'den')]
    da, db, dd = [t[k][1:] - t[k][:-1] for k in ('vp', 'vs', 'den')]
    s2, q = np.sin(th) ** 2, (b / a) ** 2
    r = np.zeros(t['vp'].shape)
    r[1:] = 0.5 * (1 - 4 * q * s2) * dd / d + da / (2 * a * np.cos(th) ** 2) - 4 * q * s2 * db / b
    return r

def plot_dhi(e, t, near, far, crest, angles_deg, curves, field_curves):
    """The near and far stacks of the trap at one clip, and the coefficient curves."""
    fig, ax = plt.subplots(1, 3, figsize=(13, 4.3), gridspec_kw={'width_ratios': [1.3, 1.3, 1]})
    c = np.percentile(np.abs(near), 99.5)
    xc, gas = e['x'][crest] / 1e3, t['t'][t['gas'][:, crest]]
    xg = e['x'][e['gas'].any(axis=0)] / 1e3
    box = dict(facecolor=(1, 1, 1, 0.85), edgecolor='none', pad=1.5)
    for a, s, name in ((ax[0], near, 'Near stack, %.0f degrees' % NEAR_DEG), (ax[1], far, 'Far stack, %.0f degrees' % FAR_DEG)):
        show(a, s, e['x'], t['t'], name + ', one clip', clip=c)
        a.set_xlim(xg.min() - 0.35, xg.max() + 0.35)
        a.set_ylim(gas.max() + 110, gas.min() - 90)
        a.annotate('top of the gas', (xc, gas.min()), (xc - 0.3, gas.min() - 50), fontsize=8, color='r', bbox=box,
                   arrowprops=dict(arrowstyle='-', color='r'))
        a.annotate('flat spot: the contact', (xc, gas.max()), (xc - 0.3, gas.max() + 70), fontsize=8, color='r', bbox=box,
                   arrowprops=dict(arrowstyle='-', color='r'))
    for name, r in curves.items():
        ax[2].plot(angles_deg, r, lw=1, label=name + ' under the standard shale')
    for name, r in field_curves.items():
        ax[2].plot(angles_deg[::5], r[::5], 'o', ms=4, label=name)
    ax[2].axhline(0, color='k', lw=0.5)
    style(ax[2], 'AVO of the top of the sand', 'incidence angle (degrees)', 'P-P reflection coefficient')
    ax[2].legend(fontsize=7)
    fig.tight_layout()
    plt.show()

def main():
    validate(MAX_ANGLE_DEG, SANDS, ROCKS)
    top = ROCKS['shale']
    angles_deg = np.arange(0, MAX_ANGLE_DEG + 1)            # one trace a degree
    brine = avo_curve(top, ROCKS['brine sand'], angles_deg)
    report_sand('brine', brine, *intercept_gradient(brine, angles_deg))
    curves = {'brine sand': brine}
    for name in SANDS:
        r = avo_curve(top, ROCKS[name], angles_deg)
        key = name.split()[0]
        curves[name] = r
        report_sand(key, r, *intercept_gradient(r, angles_deg))
        report(key + '_acoustic_r_max', acoustic_pp(top, ROCKS[name], MAX_ANGLE_DEG))
    # self-check: at normal incidence the elastic and the acoustic coefficients are the impedance contrast
    assert abs(brine[0] - acoustic_pp(top, ROCKS['brine sand'], 0)) < 1e-9
    # the field: Gassmann puts SANDS[0]'s fill into the brine sand, and the trap holds it above the flat contact
    pay = gassmann(ROCKS['brine sand'], FILLS['brine sand'], FILLS[SANDS[0]])
    fieldnote('gassmann_vp_mps', pay[0], '%.1f')
    fieldnote('gassmann_off_standard_mps', max(abs(pay[0] - ROCKS[SANDS[0]][0]), abs(pay[1] - ROCKS[SANDS[0]][1])), '%.2e')
    e = field_earth(seal=ROCKS['shale'], upper=pay, lower=ROCKS['brine sand'])
    t = field_time(e, DT_MS, T_MAX_MS)
    near = field_image(aki_richards(t, NEAR_DEG), DT_MS, F_HZ, snr=SNR)
    far = field_image(aki_richards(t, FAR_DEG), DT_MS, F_HZ, snr=SNR, seed=12)
    crest = int(np.argmax(e['gas'].sum(axis=0)))                   # the thickest gas
    brine_leg = np.nonzero(e['res'].any(axis=0) & (e['gas'].sum(axis=0) == 0) & (e['ntg'] == 1))[0]
    down = int(brine_leg[np.argmin(np.abs(brine_leg - crest))])    # the nearest trace with sand and no gas
    field_curves = {}
    for key, ix, name in (('crest', crest, SANDS[0]), ('downdip', down, 'brine sand')):
        iz = int(np.argmax(e['res'][:, ix]))                    # the top of the sand; above it, what the earth puts there
        above, below = [(e['vp'][k, ix], e['vs'][k, ix], e['den'][k, ix]) for k in (iz - 1, iz)]
        r = avo_curve(above, below, angles_deg)
        fieldnote(key + '_off_standard_r0', r[0] - zoeppritz_pp(ROCKS['shale'], ROCKS[name], 0.0), '%.4f')
        field_curves['field, %s at %.2f km' % (key, e['x'][ix] / 1e3)] = r
        fieldnote(key + '_x_m', e['x'][ix], '%.0f')
        fieldnote(key + '_r0', r[0])
        fieldnote(key + '_r_max', r[-1])
    k = int(np.argmax(t['gas'][:, crest]))
    fieldnote('crest_far_over_near', np.abs(far[k - 3:k + 4, crest]).max() / np.abs(near[k - 3:k + 4, crest]).max())
    fieldnote('flat_spot_r0', zoeppritz_pp(pay, ROCKS['brine sand'], 0.0))
    plot_dhi(e, t, near, far, crest, angles_deg, curves, field_curves)

if __name__ == '__main__':
    main()

## Where to go next

- The course: https://ogbonlab.com/textbooks/synthetic-seismic-modeling
- More notebooks: https://github.com/mosesfalowo/ogbonlab-notebooks
- Scale up: Devito, SPECFEM3D, Madagascar (course Part 12.3)